# LAB 5 - TRUC QUAN HOA DU LIEU VOI MATPLOTLIB VA SEABORN

- Thoi gian: 2 tiet (90 phut)
- Chu de: Tiep noi EduTrack - Ve bieu do bao cao phan tich
- Muc tieu:
  - Thanh thao cac loai bieu do co ban: bar, line, pie, histogram, scatter
  - Su dung Seaborn ve cac bieu do thong ke: boxplot, heatmap, pairplot
  - Dinh dang bieu do chuyen nghiep: tieu de, nhan truc, legend, mau sac
  - Tao layout nhieu bieu do tren mot hinh (subplots)

Luu y:
- Chay BUOC 0 truoc. No tai lai toan bo du lieu tu Lab4.
- Cac o co nhan [TODO] la phan ban phai tu viet code.
- Moi bieu do can co: tieu de, nhan truc x/y, va du rang bieu do hien thi ro rang.


## BUOC 0 - KHOI TAO DU LIEU (CHAY BUOC NAY TRUOC)

Copy nguyen cell khoi tao du lieu tu Lab4 vao day va chay lai.

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Cai dat phong chu ho tro tieng Viet (neu co)
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid', palette='muted')

np.random.seed(42)

# --- TAO LAI 4 BANG DU LIEU ---
df_khoahoc = pd.DataFrame({
    'ma_khoa': ['KH001', 'KH002', 'KH003', 'KH004', 'KH005'],
    'ten_khoa': [
        'Python Co Ban', 'Machine Learning', 'Web Django',
        'SQL & Database', 'Data Visualization'
    ],
    'linh_vuc': ['Lap trinh', 'AI', 'Lap trinh', 'Database', 'AI'],
    'hoc_phi': [1500000, 3500000, 2000000, 1800000, 2500000],
    'so_buoi': [12, 20, 15, 10, 12],
    'ngay_khai_giang': pd.to_datetime([
        '2024-01-10', '2024-02-01', '2024-01-20',
        '2024-03-05', '2024-02-15'
    ])
})

ho = ['Nguyen', 'Tran', 'Le', 'Pham', 'Hoang', 'Bui', 'Do', 'Ngo', 'Duong', 'Ly']
ten = ['An', 'Binh', 'Cuong', 'Dung', 'Em', 'Phong', 'Giang', 'Hai', 'Lan', 'Mai']
tinh = ['Ha Noi', 'TP HCM', 'Da Nang', 'Can Tho', 'Hue']
n_hv = 30

df_hocvien = pd.DataFrame({
    'ma_hv': [f'HV{str(i).zfill(3)}' for i in range(1, n_hv + 1)],
    'ho_ten': [f"{np.random.choice(ho)} {np.random.choice(ten)}" for _ in range(n_hv)],
    'tinh_thanh': np.random.choice(tinh, n_hv),
    'nam_sinh': np.random.randint(1995, 2005, n_hv),
    'ngay_dang_ky_he_thong': pd.to_datetime(
        np.random.choice(pd.date_range('2023-06-01', '2024-01-01'), n_hv)
    )
})

records = []
ma_phieu = 1
for i, row in df_hocvien.iterrows():
    n_dk = np.random.choice([1, 2, 3], p=[0.4, 0.4, 0.2])
    khoas = np.random.choice(df_khoahoc['ma_khoa'], n_dk, replace=False)
    for k in khoas:
        records.append({
            'ma_phieu': f'PH{str(ma_phieu).zfill(4)}',
            'ma_hv': row['ma_hv'],
            'ma_khoa': k,
            'ngay_dang_ky': pd.Timestamp('2024-01-01') + pd.Timedelta(days=int(np.random.randint(0, 90))),
            'trang_thai': np.random.choice(['Dang hoc', 'Hoan thanh', 'Bo giang giua chung'], p=[0.4, 0.5, 0.1])
        })
        ma_phieu += 1

df_dangky = pd.DataFrame(records)

tien_trinh_records = []
for _, dk in df_dangky.iterrows():
    so_buoi = df_khoahoc.loc[df_khoahoc['ma_khoa'] == dk['ma_khoa'], 'so_buoi'].values[0]
    if dk['trang_thai'] == 'Hoan thanh':
        so_buoi_da_hoc = so_buoi
    elif dk['trang_thai'] == 'Bo giang giua chung':
        so_buoi_da_hoc = int(np.random.randint(1, so_buoi // 2))
    else:
        so_buoi_da_hoc = int(np.random.randint(so_buoi // 2, so_buoi))
    for buoi in range(1, so_buoi_da_hoc + 1):
        tien_trinh_records.append({
            'ma_phieu': dk['ma_phieu'],
            'buoi_so': buoi,
            'ngay_hoc': dk['ngay_dang_ky'] + pd.Timedelta(days=buoi * 3),
            'diem_bai_tap': round(np.random.uniform(4.0, 10.0), 1),
            'tham_du': np.random.choice([True, False], p=[0.85, 0.15])
        })

df_tientrinh = pd.DataFrame(tien_trinh_records)

# Tao bang tong hop de dung chung
df_tong_hop = pd.merge(df_dangky, df_hocvien[['ma_hv', 'ho_ten', 'tinh_thanh', 'nam_sinh']], on='ma_hv', how='inner')
df_tong_hop = pd.merge(df_tong_hop, df_khoahoc[['ma_khoa', 'ten_khoa', 'hoc_phi', 'linh_vuc', 'so_buoi']], on='ma_khoa', how='inner')
df_tong_hop['ngay_dang_ky'] = pd.to_datetime(df_tong_hop['ngay_dang_ky'])
df_tong_hop['thang'] = df_tong_hop['ngay_dang_ky'].dt.to_period('M')

print("Khoi tao du lieu thanh cong!")
print(f"  df_tong_hop: {df_tong_hop.shape}")
print(f"  df_tientrinh: {df_tientrinh.shape}")

---
## PHAN 1 - BIEU DO COT (BAR CHART)

Bieu do cot dung de so sanh gia tri giua cac nhom / danh muc.


### Huong dan 1.1 - Bieu do cot doc: So dang ky theo ten khoa

In [ ]:
so_dk_khoa = df_tong_hop.groupby('ten_khoa')['ma_phieu'].count().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(so_dk_khoa.index, so_dk_khoa.values, color='#2E75B6', edgecolor='white')

# Them nhan so tren moi cot
for bar in bars:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.2,
        str(int(bar.get_height())),
        ha='center', va='bottom', fontsize=10
    )

ax.set_title('So luot dang ky theo khoa hoc', fontsize=14, pad=12)
ax.set_xlabel('Ten khoa hoc', fontsize=11)
ax.set_ylabel('So luot dang ky', fontsize=11)
ax.tick_params(axis='x', rotation=15)
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig('bieu_do_1_so_dk_khoa.png', dpi=150, bbox_inches='tight')
plt.show()

### [TODO] 1.2 - Bieu do cot ngang: Doanh thu theo tinh thanh

Ve bieu do cot nam ngang (barh) the hien tong hoc_phi theo tung tinh_thanh.
- Sap xep tang dan (de cot cao nhat hien tren cung)
- Them nhan gia tri o cuoi moi thanh
- Dat tieu de: 'Doanh thu theo tinh/thanh pho'
- Dinh dang truc x hien thi don vi trieu dong (chia cho 1,000,000)

In [ ]:
# TODO 1.2
# Goi y nhan gia tri: ax.text(gia_tri + offset, vi_tri_y, str(gia_tri), va='center')



### [TODO] 1.3 - Bieu do cot nhom: So hoc vien theo trang thai va linh vuc

Ve bieu do cot nhom (grouped bar chart) the hien so luot dang ky theo `trang_thai`, tach rieng tung `linh_vuc`.

Goi y:
- Tao pivot table: `index='trang_thai'`, `columns='linh_vuc'`, `values='ma_phieu'`, `aggfunc='count'`
- Goi `.plot(kind='bar', ax=ax)` len pivot table do

In [ ]:
# TODO 1.3



---
## PHAN 2 - BIEU DO DUONG (LINE CHART)

Bieu do duong dung de the hien xu huong thay doi theo thoi gian.


### Huong dan 2.1 - Bieu do duong doanh thu theo thang

In [ ]:
dt_thang = df_tong_hop.groupby('thang')['hoc_phi'].sum()
dt_thang.index = dt_thang.index.astype(str)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(dt_thang.index, dt_thang.values / 1e6,
        marker='o', linewidth=2.5, color='#375623', markersize=7)
ax.fill_between(dt_thang.index, dt_thang.values / 1e6, alpha=0.15, color='#375623')

# Them nhan tren moi diem
for x, y in zip(dt_thang.index, dt_thang.values / 1e6):
    ax.annotate(f'{y:.1f}M', (x, y), textcoords='offset points',
                xytext=(0, 8), ha='center', fontsize=9)

ax.set_title('Doanh thu dang ky theo thang (trieu dong)', fontsize=14, pad=12)
ax.set_xlabel('Thang', fontsize=11)
ax.set_ylabel('Doanh thu (trieu dong)', fontsize=11)
ax.tick_params(axis='x', rotation=30)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('bieu_do_2_doanh_thu_thang.png', dpi=150, bbox_inches='tight')
plt.show()

### [TODO] 2.2 - Nhieu duong tren cung mot bieu do: Doanh thu theo thang tung linh vuc

Ve bieu do duong gom 3 duong, moi duong la 1 linh_vuc (AI, Database, Lap trinh).

Goi y:
- Tao pivot table: `index='thang'`, `columns='linh_vuc'`, `values='hoc_phi'`, `aggfunc='sum'`, `fill_value=0`
- Lap qua tung cot cua pivot va goi `ax.plot()` moi cot mot lan
- Them `ax.legend()` de phan biet cac duong

In [ ]:
# TODO 2.2



### [TODO] 2.3 - Bieu do duong luy ke

Ve bieu do duong the hien doanh thu luy ke theo thang.
- Tinh doanh thu theo thang roi dung `.cumsum()`
- Them vung to mau duoi duong (fill_between)
- Tieu de: 'Doanh thu luy ke theo thang'

In [ ]:
# TODO 2.3



---
## PHAN 3 - BIEU DO TRON (PIE / DONUT CHART)

Bieu do tron the hien ti le phan bo cua mot bien phan loai.


### Huong dan 3.1 - Bieu do tron ti le trang thai dang ky

In [ ]:
tt_count = df_tong_hop['trang_thai'].value_counts()

colors = ['#2E75B6', '#70AD47', '#FF0000']
explode = [0.05] * len(tt_count)

fig, ax = plt.subplots(figsize=(7, 6))
wedges, texts, autotexts = ax.pie(
    tt_count.values,
    labels=tt_count.index,
    autopct='%1.1f%%',
    colors=colors,
    explode=explode,
    startangle=90
)
for t in autotexts:
    t.set_fontsize(11)

ax.set_title('Ti le trang thai dang ky khoa hoc', fontsize=14, pad=12)
plt.tight_layout()
plt.show()

### [TODO] 3.2 - Bieu do donut: Ti le doanh thu theo linh vuc

Ve bieu do doughnut (bieu do tron co lo giua) the hien ti le tong hoc_phi theo linh_vuc.

Goi y: Sau khi ve pie chart binh thuong, them vong tron trang o giua:
```python
circle = plt.Circle((0, 0), 0.55, color='white')
ax.add_artist(circle)
```

In [ ]:
# TODO 3.2



---
## PHAN 4 - BIEU DO PHAN PHOI (HISTOGRAM & KDE)

Dung de hieu hinh dang phan phoi cua bien so.


### Huong dan 4.1 - Histogram diem bai tap voi Seaborn

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(
    data=df_tientrinh,
    x='diem_bai_tap',
    bins=15,
    kde=True,
    color='#843C0C',
    ax=ax
)
ax.set_title('Phan phoi diem bai tap toan he thong', fontsize=13)
ax.set_xlabel('Diem bai tap', fontsize=11)
ax.set_ylabel('So luong buoi hoc', fontsize=11)
ax.axvline(df_tientrinh['diem_bai_tap'].mean(), color='navy',
           linestyle='--', linewidth=1.5, label=f"Trung binh: {df_tientrinh['diem_bai_tap'].mean():.2f}")
ax.legend()
plt.tight_layout()
plt.show()

### [TODO] 4.2 - Histogram so sanh diem theo trang thai

Su dung `sns.histplot` ve histogram diem_bai_tap, to mau khac nhau theo trang_thai cua phieu dang ky.

Goi y:
- Merge `df_tientrinh` voi `df_dangky` tren `ma_phieu` de co cot `trang_thai`
- Dung tham so `hue='trang_thai'` trong `sns.histplot`
- Dat `alpha=0.5` de thay duoc cac duong chong nhau

In [ ]:
# TODO 4.2



### [TODO] 4.3 - KDE Plot: Phan phoi nam sinh hoc vien theo tinh thanh

Su dung `sns.kdeplot` ve duong phan phoi nam_sinh cua hoc vien, phan biet theo tinh_thanh (moi tinh 1 duong).

In [ ]:
# TODO 4.3
# Goi y: sns.kdeplot(data=df_hocvien, x='nam_sinh', hue='tinh_thanh', fill=True, alpha=0.3)



---
## PHAN 5 - BOXPLOT VA VIOLIN PLOT

Boxplot the hien phan phoi (min, Q1, median, Q3, max) va phat hien outlier. Violin plot them thong tin ve hinh dang phan phoi.


### Huong dan 5.1 - Boxplot diem bai tap theo khoa hoc

In [ ]:
# Merge de co ten khoa trong tientrinh
df_tt_khoa = pd.merge(
    df_tientrinh,
    df_dangky[['ma_phieu', 'ma_khoa']],
    on='ma_phieu'
)
df_tt_khoa = pd.merge(
    df_tt_khoa,
    df_khoahoc[['ma_khoa', 'ten_khoa']],
    on='ma_khoa'
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(
    data=df_tt_khoa,
    x='ten_khoa',
    y='diem_bai_tap',
    palette='Blues',
    ax=ax
)
ax.set_title('Phan phoi diem bai tap theo khoa hoc', fontsize=13)
ax.set_xlabel('Ten khoa', fontsize=11)
ax.set_ylabel('Diem bai tap', fontsize=11)
ax.tick_params(axis='x', rotation=20)
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

### [TODO] 5.2 - Violin Plot diem theo linh vuc

Ve violin plot the hien phan phoi `diem_bai_tap` theo `linh_vuc`.

Goi y: `sns.violinplot(data=..., x='linh_vuc', y='diem_bai_tap', inner='quartile')`

In [ ]:
# TODO 5.2
# Can merge df_tt_khoa voi df_khoahoc de co cot linh_vuc (neu chua co)



### [TODO] 5.3 - Boxplot so sanh diem giua hoc vien tham du va vang mat

Ve boxplot so sanh `diem_bai_tap` giua 2 nhom: `tham_du=True` va `tham_du=False`.
- Dat mau xanh la cho nhom tham du, mau do nhat cho nhom vang mat
- Them duong trung binh toan bo len bieu do bang `ax.axhline()`

In [ ]:
# TODO 5.3



---
## PHAN 6 - HEATMAP VA SCATTER PLOT


### Huong dan 6.1 - Heatmap so luot dang ky theo tinh va khoa

In [ ]:
pivot_heat = pd.pivot_table(
    df_tong_hop,
    values='ma_phieu',
    index='tinh_thanh',
    columns='ten_khoa',
    aggfunc='count',
    fill_value=0
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(
    pivot_heat,
    annot=True,
    fmt='d',
    cmap='YlOrRd',
    linewidths=0.5,
    ax=ax
)
ax.set_title('So luot dang ky theo tinh thanh va khoa hoc', fontsize=13)
ax.set_xlabel('Khoa hoc', fontsize=11)
ax.set_ylabel('Tinh/Thanh pho', fontsize=11)
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

### [TODO] 6.2 - Heatmap ty le hoan thanh theo tinh va linh vuc

Tao pivot table the hien ty le % hoc vien Hoan thanh (so voi tong dang ky) theo tinh_thanh va linh_vuc.
Dung cmap='Greens', them chu thich `%` trong cac o bang `fmt='.0f'` va nhan 100.

In [ ]:
# TODO 6.2
# Goi y:
# df_tong_hop['hoan_thanh'] = (df_tong_hop['trang_thai'] == 'Hoan thanh').astype(int)
# roi dung pivot_table voi aggfunc='mean' va nhan 100



### [TODO] 6.3 - Scatter Plot: Moi quan he giua so_buoi va hoc_phi

Ve scatter plot tu `df_khoahoc`, truc x la `so_buoi`, truc y la `hoc_phi`.
- To mau diem theo `linh_vuc` (dung vong lap hoac `hue` cua seaborn)
- Them ten khoa ben canh moi diem (`ax.annotate()`)
- Tieu de: 'Moi quan he giua so buoi hoc va hoc phi'

In [ ]:
# TODO 6.3



---
## PHAN 7 - SUBPLOT: NHIEU BIEU DO TREN MOT HINH

`plt.subplots(n_rows, n_cols)` tao luoi cac o bieu do. Rat huu ich khi can tao dashboard hoac bao cao tong hop.


### Huong dan 7.1 - Dashboard 2x2

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Bao cao tong quan EduTrack', fontsize=16, fontweight='bold', y=1.01)

# O [0,0]: So dang ky theo khoa
ax = axes[0, 0]
so_dk = df_tong_hop.groupby('ten_khoa')['ma_phieu'].count().sort_values(ascending=True)
ax.barh(so_dk.index, so_dk.values, color='#2E75B6')
ax.set_title('So dang ky theo khoa')
ax.set_xlabel('So luot')

# O [0,1]: Ti le trang thai
ax = axes[0, 1]
tt = df_tong_hop['trang_thai'].value_counts()
ax.pie(tt.values, labels=tt.index, autopct='%1.1f%%',
       colors=['#70AD47', '#2E75B6', '#FF6B6B'])
ax.set_title('Ti le trang thai dang ky')

# O [1,0]: Doanh thu theo thang
ax = axes[1, 0]
dt = df_tong_hop.groupby('thang')['hoc_phi'].sum()
dt.index = dt.index.astype(str)
ax.plot(dt.index, dt.values / 1e6, marker='o', color='#375623')
ax.set_title('Doanh thu theo thang (tr.dong)')
ax.tick_params(axis='x', rotation=30)
ax.grid(alpha=0.4)

# O [1,1]: Boxplot diem theo linh vuc
ax = axes[1, 1]
df_tt_lv = pd.merge(df_tientrinh, df_dangky[['ma_phieu', 'ma_khoa']], on='ma_phieu')
df_tt_lv = pd.merge(df_tt_lv, df_khoahoc[['ma_khoa', 'linh_vuc']], on='ma_khoa')
sns.boxplot(data=df_tt_lv, x='linh_vuc', y='diem_bai_tap', palette='pastel', ax=ax)
ax.set_title('Phan phoi diem theo linh vuc')
ax.set_xlabel('')

plt.tight_layout()
plt.savefig('dashboard_edutrack.png', dpi=150, bbox_inches='tight')
plt.show()
print('Da luu: dashboard_edutrack.png')

### [TODO] 7.2 - Tao dashboard 2x3 (6 bieu do)

Tao `fig, axes = plt.subplots(2, 3, figsize=(16, 10))` va dien vao 6 o theo noi dung sau:

| Vi tri | Noi dung | Loai bieu do |
|--------|----------|--------------|
| [0,0]  | Doanh thu theo linh vuc | Bar chart |
| [0,1]  | So hoc vien theo tinh thanh | Bar chart ngang |
| [0,2]  | Ty le doanh thu theo linh vuc | Pie chart |
| [1,0]  | Phan phoi diem bai tap | Histogram + KDE |
| [1,1]  | Doanh thu luy ke theo thang | Line chart |
| [1,2]  | Diem trung binh theo buoi so (5 buoi dau) | Bar chart |

Luu hinh voi ten `dashboard_tong_hop.png`.

In [ ]:
# TODO 7.2



---
## PHAN 8 - BAI TAP TONG HOP CUOI LAB

Day la bai tap ket hop tat ca ky nang tu Lab4 va Lab5. Lam theo thu tu.


### [TODO] 8.1 - Tinh ty le tham du theo khoa va ve bieu do

Tu `df_tientrinh` da merge voi ten khoa:
1. Tinh ty le tham_du trung binh (True = 1, False = 0) theo tung `ten_khoa`
2. Nhan 100 de ra %
3. Ve bar chart the hien ty le tham du (%)
4. Them duong ngang `ax.axhline(y=85, ...)` danh dau nguong muc tieu 85%

In [ ]:
# TODO 8.1



### [TODO] 8.2 - Phat hien khoa hoc co xu huong diem giam dan

Tu `df_tientrinh` (da merge ten khoa):
1. Loc ra khoa 'Machine Learning' (khoa co nhieu buoi nhat - 20 buoi)
2. Tinh diem_bai_tap trung binh theo `buoi_so`
3. Ve bieu do duong the hien diem trung binh theo thu tu buoi hoc
4. Them duong xu huong (trend line) bang `np.polyfit` bac 1
5. Nhan xet: diem co xu huong tang hay giam?

In [ ]:
# TODO 8.2
# Goi y them trend line:
# z = np.polyfit(x_values, y_values, 1)  # bac 1 = duong thang
# p = np.poly1d(z)
# ax.plot(x_values, p(x_values), 'r--', label='Xu huong')



### [TODO] 8.3 - Bao cao tong ket dang Excel nhieu sheet

Xuat file `bao_cao_lab5.xlsx` gom 4 sheet:
- `TongHop`: df_tong_hop (khong co cot thang - Period object se bi loi Excel)
- `ThongKe_Khoa`: so_dang_ky, tong_doanh_thu, ty_le_hoan_thanh theo ten_khoa
- `TyLe_ThamDu`: ty le tham du theo ten_khoa
- `DoanhThu_Thang`: doanh thu tong va luy ke theo thang

Sau khi xuat, in ra kich thuoc tung sheet.

In [ ]:
# TODO 8.3
# Luu y: Truoc khi to_excel, doi cot kieu Period sang string:
# df['thang'] = df['thang'].astype(str)



---
## TOM TAT LAB 5

Trong lab nay ban da luyen tap:

1. Bieu do cot (doc va ngang), them nhan gia tri tren cot
2. Bieu do cot nhom so sanh nhieu nhom cung luc
3. Bieu do duong xu huong theo thoi gian, nhieu duong cung mot hinh
4. Bieu do tron va doughnut the hien ti le
5. Histogram va KDE phan tich phan phoi
6. Boxplot va Violin Plot phat hien outlier va so sanh phan phoi
7. Heatmap va Scatter Plot xem moi quan he 2 chieu
8. Subplot tao dashboard nhieu bieu do tren mot hinh
9. Xuat ket qua ra file PNG va Excel nhieu sheet

Ket thuc: Luu notebook va dam bao Kernel > Restart & Run All chay khong loi.
